# 03 - Time-Series Feature Engineering & Leakage Auditing

## Objectives
1. Ingest clean daily product demand series (`data/processed/daily_product_demand.parquet`).
2. Construct autoregressive lags ($t-1, t-7, t-14, t-28$) independently per product.
3. Construct rolling statistics (7, 14, 28 days) strictly shifted by 1 row to prevent lookahead leakage.
4. Extract calendar attributes (`day_of_week`, `month`, `day_of_month`, `is_weekend`).
5. Differentiate between un-modeled warm-up history and model-eligible rows (`is_model_eligible`).

In [ ]:
import pandas as pd
from pathlib import Path
from retail_forecasting.features.build_features import build_feature_pipeline, analyze_product_histories

# Load daily demand
data_path = Path("../data/processed/daily_product_demand.parquet")
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} daily demand records.")

# Run feature engineering pipeline on a sample of high-volume products for illustration
top_products = df.groupby("product_id")["demand"].sum().nlargest(10).index
sample_df = df[df["product_id"].isin(top_products)].copy()

features_df = build_feature_pipeline(sample_df)
print(f"Generated features shape: {features_df.shape}")
print(f"Model-eligible records: {features_df['is_model_eligible'].sum():,} / {len(features_df):,}")
features_df.head(10)

### Verification of Leak-Free Mechanics
- At date $t$, `lag_1` is strictly demand at $t-1$.
- At date $t$, `rolling_mean_7` is the mean of demand from $t-7$ to $t-1$.
- Missing historical lags are retained as `NaN` (never silently zero-filled).